# Ingestion, Exploration (EDA) & Préparation des Données de LogiDistrib

##  Objectif du Notebook
Ce notebook constitue la première étape technique du projet **LogiDistrib**.

###  Pourquoi cette phase préparatoire est indispensable

Avant de charger des données dans un Data Warehouse comme BigQuery, un réflexe clé en Analytics Engineering consiste à valider la solidité des fondations. 

L'objectif de cette étape est **purement préventif** : 
* **Sécuriser la qualité :** S'assurer que le jeu de données ne contient ni incohérences physiques (prix ou quantités absurdes), ni fausses promesses sur sa structure (doublons invisibles sur la clé primaire).
* **Eliminer le superflu :** Éliminer le bruit technique et les variables hors périmètre métier dès la source pour travailler sur une table sobre et parfaitement lisible.
* **Optimiser le pipeline :** Formater et exporter une donnée « propre par construction » vers BigQuery pour ne plus avoir à corriger des anomalies techniques en cours de route.

En éliminant les risques en amont, on s'assure que tout le travail SQL à venir sera exclusivement concentré sur la valeur métier : la modélisation, le calcul des KPI et l'analyse décisionnelle.

## chargement du dataset
---

In [1]:
import pandas as pd
tables = pd.read_csv("../dataset/supply_chain_dataset1.csv")
# Standardisation préventive des noms de colonnes (snake_case)
# On harmonise les noms des colonnes pour ne pas à voir à les corriger manuellement dans Bigquery
tables.columns = tables.columns.str.strip().str.lower()
tables.head()


,unnamed: 0.1,unnamed: 0,date,sku_id,warehouse_id,supplier_id,region,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price,promotion_flag,stockout_flag,demand_forecast
0,0,0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,14,379,0,13.95,20.48,0,0,8.52
1,1,1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,14,379,0,13.95,20.48,0,0,18.63
2,2,2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,14,379,0,13.95,20.48,1,0,39.62
3,3,3,2024-01-04,SKU_1,WH_1,SUP_8,South,24,516,14,379,0,13.95,20.48,0,0,19.43
4,4,4,2024-01-05,SKU_1,WH_1,SUP_8,West,21,495,14,379,0,13.95,20.48,0,0,18.70


## Diagnostic Rapide EDA
---

In [10]:
# Vue d'ensemble : dimensions, types, valeurs nulles
print(tables.info())

# Vérification des doublons stricts
print(f"Doublons stricts : {tables.duplicated().sum()}")

# Contrôle des clés : vérification d'unicité (1 ligne = 1 SKU x 1 Entrepôt x 1 Jour)
check_keys = tables.groupby(['date', 'sku_id', 'warehouse_id']).size()
print(f"Nombre de combinaisons en doublon : {(check_keys > 1).sum()}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 91250 entries, 0 to 91249
Data columns (total 17 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   unnamed: 0.1             91250 non-null  int64  
 1   unnamed: 0               91250 non-null  int64  
 2   date                     91250 non-null  object 
 3   sku_id                   91250 non-null  object 
 4   warehouse_id             91250 non-null  object 
 5   supplier_id              91250 non-null  object 
 6   region                   91250 non-null  object 
 7   units_sold               91250 non-null  int64  
 8   inventory_level          91250 non-null  int64  
 9   supplier_lead_time_days  91250 non-null  int64  
 10  reorder_point            91250 non-null  int64  
 11  order_quantity           91250 non-null  int64  
 12  unit_cost                91250 non-null  float64
 13  unit_price               91250 non-null  float64
 14  promotion_flag        

### Constats

* **Volume & Complétude :** Le jeu de données contient exactement **91 250 lignes** et **17 colonnes**, sans aucune valeur manquante (`0 non-null` sur l'ensemble des colonnes). 
* **Intégrité de la Clé Primaire :** Le test d'unicité sur la clé composite `(date, sku_id, warehouse_id)` renvoie **0 doublons**. La granularité (1 ligne = 1 produit × 1 entrepôt × 1 jour) est strictement respectée.
* **Colonnes Parasites Identifiées :** Présence de deux colonnes d'index générées à l'export (`unnamed: 0.1` et `unnamed: 0`) à supprimer.
* **Alignement avec le Cadrage :** La propreté absolue des données (absence de trous de collecte ou de doublons) confirme la nature synthétique et pédagogique du dataset Kaggle, conformément aux limites méthodologiques identifiées dans le cadrage du projet.

**Décision pour la phase de nettoyage :**
1. Convertir la colonne `date` au format `datetime`.
2. Supprimer les colonnes parasites d'indexation (`unnamed: *`).
3. Supprimer les colonnes hors périmètre métier (`supplier_id`, `region`, `promotion_flag`, `demand_forecast`).
4. Suppression de Stockout_Flag car colonne constante, aucune valeur analytique
5. Exporter le dataset nettoyé au format Parquet pour ingestion dans BigQuery.
---

## Aperçu statistque sur les colonnes utiles à l'analyse

In [11]:
# Sélection des colonnes numériques ayant un sens métier
cols_metrics = [
    'units_sold', 
    'inventory_level', 
    'supplier_lead_time_days', 
    'reorder_point', 
    'order_quantity', 
    'unit_cost', 
    'unit_price'
]

# Analyse statistique ciblée
tables[cols_metrics].describe()

,units_sold,inventory_level,supplier_lead_time_days,reorder_point,order_quantity,unit_cost,unit_price
count,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000,91250.000000
mean,20.054564,471.522312,7.984000,300.068000,19.272493,12.203320,18.261800
std,9.068602,133.488002,3.907929,54.879945,82.340831,4.574982,7.121136
min,0.000000,168.000000,2.000000,201.000000,0.000000,5.020000,6.950000
25%,13.000000,370.000000,4.000000,252.000000,0.000000,8.180000,12.000000
50%,20.000000,461.000000,8.000000,300.000000,0.000000,11.990000,18.180000
75%,27.000000,564.000000,11.000000,346.000000,0.000000,16.320000,23.390000
max,59.000000,990.000000,14.000000,398.000000,499.000000,19.760000,35.100000


###  Constat & Analyse des métriques 

L'analyse descriptive ciblée sur les variables numériques métiers confirme la bonne santé globale des données et permet d'identifier la structure des variables :

* **Coûts & Prix (`unit_cost`, `unit_price`) :** 
  * **Absence d'anomalie :** Les valeurs minimales (5,02 € pour le coût, 6,95 € pour le prix) confirment l'absence de montants négatifs ou nuls.
  * **Cohérence de marge :** Le prix moyen (18,26 €) supérieur au coût moyen (12,20 €) garantit la viabilité des calculs de valorisation financière du stock immobilisé.

* **Délai fournisseur (`supplier_lead_time_days`) :** 
  * Les délais d'approvisionnement varient entre **2 et 14 jours** (moyenne de 8 jours), offrant une distribution réaliste pour la modélisation du point de commande.

* **Ventes quotidiennes (`units_sold`) :** 
  * Les volumes s'échelonnent de **0 à 59 unités/jour** (moyenne de 20 unités/jour), sans valeurs aberrantes.

* **Spécificité Métier #1 : Niveaux de stock (`inventory_level`) :** 
  * Le minimum est de **168 unités** dans les chiffres bruts.
  * *Interprétation  :* Le stock physique enregistré au niveau de la ligne ne tombe jamais à 0. Les ruptures indiquées par `Stockout_Flag = 1` correspondent à une demande non satisfaite (ventes perdues) lorsque le niveau de stock était insuffisant face au volume de commande.

* **Spécificité Métier #2 — Ordres de commande (`order_quantity`) :** 
  * Le 75e percentile est égal à **0**. 
  * *Interprétation :*  Plus de 75% du temps, aucune commande n'est passée (fonctionnement normal au quotidien). Les réapprovisionnements se déclenchent par pics ponctuels lors du franchissement du seuil d'alerte (jusqu'à **499 unités**).

---

**Conclusion pour la suite du projet :** 
Les ordres de grandeur sont validés et ne nécessitent aucun nettoyage d'aberrations. 

### Evaluation ciblé sur Stockout_flag
---

In [2]:
# 1. Vérification des valeurs uniques (Contrôle du caractère strictement binaire)
print("Valeurs uniques de stockout_flag :", tables['stockout_flag'].unique().tolist())

# 2. Répartition du taux de rupture global
print("\nTaux de rupture global (en %) :")
print(tables['stockout_flag'].value_counts(normalize=True) * 100)

Valeurs uniques de stockout_flag : [0]

Taux de rupture global (en %) :
stockout_flag
0    100.0
Name: proportion, dtype: float64


### Constat : Fiabilité de Stockout_Flag
---
La vérification de cette colonne (`value_counts`) révèle qu'elle est à **0** sur l'intégralité du dataset, elle est donc inutilisable comme KPI 

de l'axe de vérification des ruptures constatées.

Aucune autre colonne du périmètre retenu ne permet de reconstruire ce signal de façon fiable : 
- `Units_Sold` quantifie les ventes réellement honorées, non celles insatisfaites par ruptures de stock
-  `Inventory_Level` ne descend jamais vers des valeurs proches de zéro (minimum observé : 168 unités)
-  `Demand_Forecast`, seule variable susceptible de révéler un écart demande/stock, a été volontairement exclue du périmètre au cadrage
    son format actuel ( valeur en float) est ambigu
    
**Décision** : le Niveau 1 sur les ruptures constaté est retiré de l'Axe 1 de la problématique. 
L'analyse se recentre sur les niveaux 2 (alerte) et 3 (diagnostic structurel).

### Contrôle de cohérence : Reorder_Point et Supplier_Lead_Time_Days sont-ils fixes par SKU × Entrepôt ?
---

In [6]:
# Vérification du nombre de valeurs uniques par couple (SKU x Entrepôt)
check_params = tables.groupby(['sku_id', 'warehouse_id'])[['reorder_point', 'supplier_lead_time_days']].nunique()

print("Bilan de variabilité des paramètres par couple SKU x Entrepôt :")
print(f"- Couples avec plusieurs 'reorder_point' différents : {(check_params['reorder_point'] > 1).sum()}")
print(f"- Couples avec plusieurs 'supplier_lead_time_days' différents : {(check_params['supplier_lead_time_days'] > 1).sum()}")

# Contrôle visuel sur un sous-ensemble (ex: top 5 couples)
check_params.head()

Bilan de variabilité des paramètres par couple SKU x Entrepôt :
- Couples avec plusieurs 'reorder_point' différents : 0
- Couples avec plusieurs 'supplier_lead_time_days' différents : 0


reorder_point  supplier_lead_time_days
sku_id warehouse_id                                        
SKU_1  WH_1                      1                        1
       WH_2                      1                        1
       WH_3                      1                        1
       WH_4                      1                        1
       WH_5                      1                        1

### Constat : Stabilité des paramètres de réapprovisionnement
---
La vérification par `groupby(['sku_id','warehouse_id'])[['reorder_point','supplier_lead_time_days']].nunique()` confirme qu'il s'agit bien de paramètres de configuration fixes, pas de valeurs qui bougent jour après jour.

Ces variables se comportent bien comme des paramètres de configuration fixes, conformément à l'hypothèse du cadrage.

Le Niveau 3 (diagnostic structurel) de l'Axe 1 peut donc s'appuyer dessus sans réserve.

### Vérification de la couverture temporelle : chaque SKU × Entrepôt est-il suivi sur les 365 jours ?
--

In [8]:
# 1. Nombre total de SKU et d'entrepôts uniques
n_skus = tables['sku_id'].nunique()
n_warehouses = tables['warehouse_id'].nunique()
print(f"Nombre de SKU uniques : {n_skus}")
print(f"Nombre d'entrepôts uniques : {n_warehouses}")

# 2. Vérification que CHAQUE couple (SKU x Entrepôt) possède exactement 365 jours d'historique
days_per_pair = tables.groupby(['sku_id', 'warehouse_id'])['date'].nunique()

print(f"\nNombre de couples SKU x Entrepôt testés : {len(days_per_pair)} (attendu : {n_skus * n_warehouses})")
print(f"Nombre de couples n'ayant PAS exactement 365 jours : {(days_per_pair != 365).sum()}")

Nombre de SKU uniques : 50
Nombre d'entrepôts uniques : 5

Nombre de couples SKU x Entrepôt testés : 250 (attendu : 250)
Nombre de couples n'ayant PAS exactement 365 jours : 0


### Constats

La vérification confirme un référentiel stable et complet sur l'ensemble de la période :

* **50 SKU uniques** et **5 entrepôts uniques**, conformes aux volumes annoncés dans le cadrage.
* **250 couples SKU × Entrepôt** (50 × 5) tous présents dans les données donc aucune combinaison n'est totalement absente du fichier.
* **0 couple avec une couverture incomplète** — chacun des 250 couples est suivi sur exactement 365 jours, sans trou ni doublon dans la série temporelle.

**Conclusion :** le référentiel est fiable de bout en bout. Aucun SKU n'a été introduit ou retiré en cours d'année, aucun entrepôt n'a été ajouté

a posteriori. Cette stabilité conforte la solidité des futurs calculs de moyennes et de tendances (Axe 2 notamment), qui supposent une série 

continue par couple.

In [9]:
# 1. Conversion de la date
tables['date'] = pd.to_datetime(tables['date'])

# 2. Suppression des colonnes hors périmètre et des index parasites
cols_to_drop = [
    'unnamed: 0.1', 'unnamed: 0', 
    'region', 'supplier_id', 'promotion_flag', 'demand_forecast', 'stockout_flag'
]
# stockout_flag est retiré car il n'est plus analytiquement pertinent
tables_clean = tables.drop(columns=[c for c in cols_to_drop if c in tables.columns])

# 3. Export Parquet
tables_clean.to_parquet("../dataset/processed/clean_logidistrib_inventory.parquet", index=False)
print("Nettoyage terminé et fichier Parquet généré avec succès !")

Nettoyage terminé et fichier Parquet généré avec succès !
